# Pipeline Maintenance Script

This notebook performs periodic maintenance operations on the **project_voltstream** pipeline tables:

**Operations:**
* **OPTIMIZE**: Compacts small files and applies Z-ordering on specified columns for improved query performance
* **VACUUM**: Removes old data files no longer referenced by the Delta table (30-day retention)

**Target Tables:**

*Silver Layer - OPTIMIZE + VACUUM:*
* `weather_data` - Z-ordered by weather_sk, lat, lon
* `ev_stations` - Z-ordered by station_sk, station_id, latitude, longitude
* `ev_connections` - Z-ordered by connection_sk, station_id

*Silver Layer - VACUUM ONLY:*
* `quarantined_stations` - No optimization, vacuum only
* `quarantined_connectors` - No optimization, vacuum only

*Gold Layer - OPTIMIZE + VACUUM:*
* `station_dim` - Z-ordered by station_id, latitude, longitude
* `station_facts` - Z-ordered by station_id, weather_sk
* `weather_dim` - Z-ordered by weather_sk, weather_zone_id

**Note**: Bronze tables are excluded from maintenance operations.

**Schedule**: Run this notebook monthly or as needed to maintain optimal table performance.

In [0]:
import sys
from datetime import datetime
from pyspark.sql import SparkSession
import logging
import uuid

# Add logger folder to path
sys.path.append("/Workspace/Users/nseekeminiudo@gmail.com/project_volltstream")

# Import custom logger
from logger.custom_logging import set_up_logger, get_job_logger

# Set up custom logger
base_logger = set_up_logger(log_to_file=False)
run_id = str(uuid.uuid4())

# Get job-specific logger
log = get_job_logger(
    base_logger, layer="maintenance", job="pipeline_maintenance", run_id=run_id
)

log(logging.INFO, "Pipeline maintenance script initialized")
log(
    logging.INFO,
    f"Execution started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
)
log(logging.INFO, f"Run ID: {run_id}")

# Get Spark session
spark = SparkSession.builder.getOrCreate()

In [0]:
# Configuration for tables to maintain
# Format: {"table_name": ["zorder_col1", "zorder_col2", ...]}

# Tables that get OPTIMIZE + VACUUM
TABLES_CONFIG = {
    # Silver layer tables
    "silver_dev.electrovolt.weather_data": ["weather_sk", "lat", "lon"],
    "silver_dev.electrovolt.ev_stations": [
        "station_sk",
        "station_id",
        "latitude",
        "longitude",
    ],
    "silver_dev.electrovolt.ev_connections": ["connection_sk", "station_id"],
    # Gold layer tables
    "gold_dev.electrovolt.station_dim": ["station_id", "latitude", "longitude"],
    "gold_dev.electrovolt.station_facts": ["station_id", "weather_sk"],
    "gold_dev.electrovolt.weather_dim": ["weather_sk", "weather_zone_id"],
}

# Tables that get VACUUM ONLY (no optimization)
VACUUM_ONLY_TABLES = [
    "silver_dev.electrovolt.quarantined_stations",
    "silver_dev.electrovolt.quarantined_connectors",
]

# Vacuum retention period (in hours)
# 30 days = 720 hours
VACUUM_RETENTION_HOURS = 720

log(logging.INFO, f"Configured {len(TABLES_CONFIG)} table(s) for OPTIMIZE + VACUUM")
log(logging.INFO, f"Configured {len(VACUUM_ONLY_TABLES)} table(s) for VACUUM ONLY")
log(
    logging.INFO,
    f"Vacuum retention period: {VACUUM_RETENTION_HOURS} hours ({VACUUM_RETENTION_HOURS/24} days)",
)

log(logging.INFO, "\nOPTIMIZE + VACUUM tables:")
for table, zorder_cols in TABLES_CONFIG.items():
    log(logging.INFO, f"  - {table}: Z-order columns = {', '.join(zorder_cols)}")

log(logging.INFO, "\nVACUUM ONLY tables:")
for table in VACUUM_ONLY_TABLES:
    log(logging.INFO, f"  - {table}")

In [0]:
def optimize_table(table_name, zorder_columns):
    """
    Optimize a Delta table with Z-ordering on specified columns.

    Args:
        table_name (str): Fully qualified table name (catalog.schema.table)
        zorder_columns (list): List of column names to Z-order by

    Returns:
        dict: Status and metrics from the operation
    """
    start_time = datetime.now()
    log(logging.INFO, f"Starting OPTIMIZE for {table_name}", dataset=table_name)

    try:
        # Build OPTIMIZE command with Z-ORDER BY
        zorder_clause = ", ".join(zorder_columns)
        optimize_sql = f"OPTIMIZE {table_name} ZORDER BY ({zorder_clause})"

        log(logging.INFO, f"Executing: {optimize_sql}", dataset=table_name)
        result = spark.sql(optimize_sql)

        # Display optimization metrics
        display(result)

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        log(
            logging.INFO,
            f"OPTIMIZE completed for {table_name} in {duration:.2f} seconds",
            dataset=table_name,
        )

        return {
            "status": "success",
            "table": table_name,
            "operation": "optimize",
            "duration_seconds": duration,
            "timestamp": end_time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e)

        log(
            logging.ERROR,
            f"OPTIMIZE failed for {table_name}: {error_msg}",
            dataset=table_name,
        )

        return {
            "status": "failed",
            "table": table_name,
            "operation": "optimize",
            "error": error_msg,
            "duration_seconds": duration,
            "timestamp": end_time.strftime("%Y-%m-%d %H:%M:%S"),
        }


log(logging.INFO, "Optimize function defined")

In [0]:
def vacuum_table(table_name, retention_hours):
    """
    Vacuum a Delta table to remove old data files.

    Args:
        table_name (str): Fully qualified table name (catalog.schema.table)
        retention_hours (int): Retention period in hours (files older than this are removed)

    Returns:
        dict: Status and metrics from the operation
    """
    start_time = datetime.now()
    log(
        logging.INFO,
        f"Starting VACUUM for {table_name} with {retention_hours}h retention",
        dataset=table_name,
    )

    try:
        # Build VACUUM command with retention period
        vacuum_sql = f"VACUUM {table_name} RETAIN {retention_hours} HOURS"

        log(logging.INFO, f"Executing: {vacuum_sql}", dataset=table_name)
        result = spark.sql(vacuum_sql)

        # Display vacuum metrics
        display(result)

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        log(
            logging.INFO,
            f"VACUUM completed for {table_name} in {duration:.2f} seconds",
            dataset=table_name,
        )

        return {
            "status": "success",
            "table": table_name,
            "operation": "vacuum",
            "retention_hours": retention_hours,
            "duration_seconds": duration,
            "timestamp": end_time.strftime("%Y-%m-%d %H:%M:%S"),
        }

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e)

        log(
            logging.ERROR,
            f"VACUUM failed for {table_name}: {error_msg}",
            dataset=table_name,
        )

        return {
            "status": "failed",
            "table": table_name,
            "operation": "vacuum",
            "error": error_msg,
            "duration_seconds": duration,
            "timestamp": end_time.strftime("%Y-%m-%d %H:%M:%S"),
        }


log(logging.INFO, "Vacuum function defined")

In [0]:
# Track all operation results
results = []

# ========== Process OPTIMIZE + VACUUM tables ==========
log(logging.INFO, "\n" + "=" * 80)
log(logging.INFO, "PHASE 1: OPTIMIZE + VACUUM")
log(logging.INFO, "=" * 80)

for table_name, zorder_cols in TABLES_CONFIG.items():
    log(logging.INFO, f"\n{'='*60}")
    log(logging.INFO, f"Processing table: {table_name}", dataset=table_name)
    log(logging.INFO, f"{'='*60}")

    # Step 1: Optimize with Z-ordering
    optimize_result = optimize_table(table_name, zorder_cols)
    results.append(optimize_result)

    # Step 2: Vacuum (only if optimize succeeded)
    if optimize_result["status"] == "success":
        vacuum_result = vacuum_table(table_name, VACUUM_RETENTION_HOURS)
        results.append(vacuum_result)
    else:
        log(
            logging.WARNING,
            f"Skipping VACUUM for {table_name} due to OPTIMIZE failure",
            dataset=table_name,
        )

# ========== Process VACUUM ONLY tables ==========
log(logging.INFO, "\n" + "=" * 80)
log(logging.INFO, "PHASE 2: VACUUM ONLY")
log(logging.INFO, "=" * 80)

for table_name in VACUUM_ONLY_TABLES:
    log(logging.INFO, f"\n{'='*60}")
    log(
        logging.INFO,
        f"Processing table: {table_name} (VACUUM ONLY)",
        dataset=table_name,
    )
    log(logging.INFO, f"{'='*60}")

    vacuum_result = vacuum_table(table_name, VACUUM_RETENTION_HOURS)
    results.append(vacuum_result)

# ========== Summary ==========
log(logging.INFO, f"\n{'='*80}")
log(logging.INFO, "MAINTENANCE SUMMARY")
log(logging.INFO, f"{'='*80}")

total_operations = len(results)
success_count = sum(1 for r in results if r["status"] == "success")
failed_count = total_operations - success_count

log(logging.INFO, f"Total operations: {total_operations}")
log(logging.INFO, f"Successful: {success_count}")
log(logging.INFO, f"Failed: {failed_count}")

if failed_count > 0:
    log(logging.WARNING, "\nFailed operations:")
    for r in results:
        if r["status"] == "failed":
            log(
                logging.WARNING,
                f"  - {r['table']} ({r['operation']}): {r.get('error', 'Unknown error')}",
                dataset=r["table"],
            )

log(
    logging.INFO,
    f"\nMaintenance script completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
)
log(logging.INFO, f"Run ID: {run_id}")

# Display results as DataFrame for easy review
import pandas as pd

results_df = pd.DataFrame(results)
display(results_df)